In [1]:
!pip install matplotlib seaborn -q

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("olist_queries") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 14:51:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/12 14:51:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Sử dụng Spark để query các thông tin sau
1. Top 3 danh mục sản phẩm bán chạy nhất tại mỗi bang
2. Tính thời gian giao hàng trung bình theo từng tháng và so sánh với thời gian dự kiến để xem tháng nào logistics của Olist bị trễ nhiều nhất
3. Truy vấn danh sách các khách hàng có chi tiêu cao hơn mức chi tiêu trung bình của toàn bộ khách hàng trong cùng bang của họ.

In [5]:
HDFS_PATH = "hdfs://namenode:9000/bigdata/"

In [6]:
orders = spark.read.csv(HDFS_PATH + "olist_orders_dataset.csv", header=True, inferSchema=True)
items = spark.read.csv(HDFS_PATH + "olist_order_items_dataset.csv", header=True, inferSchema=True)
payments = spark.read.csv(HDFS_PATH + "olist_order_payments_dataset.csv", header=True, inferSchema=True)
customers = spark.read.csv(HDFS_PATH + "olist_customers_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv(HDFS_PATH + "olist_products_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv(HDFS_PATH + "olist_order_reviews_dataset.csv", header=True, inferSchema=True)

In [16]:
orders.cache()

DataFrame[order_id: string, customer_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp]

In [7]:
orders.createOrReplaceTempView("orders")
items.createOrReplaceTempView("order_items")
payments.createOrReplaceTempView("payments")
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
reviews.createOrReplaceTempView("reviews")

## Top 3 danh mục sản phẩm bán chạy nhất tại mỗi bang

In [8]:
q1_sql = spark.sql("""
    WITH category_revenue AS (
        SELECT 
            c.customer_state,
            p.product_category_name,
            SUM(i.price) as total_sales,
            DENSE_RANK() OVER (PARTITION BY c.customer_state ORDER BY SUM(i.price) DESC) as rank
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN order_items i ON o.order_id = i.order_id
        JOIN products p ON i.product_id = p.product_id
        WHERE o.order_status = 'delivered' AND p.product_category_name IS NOT NULL
        GROUP BY c.customer_state, p.product_category_name
    )
    SELECT * FROM category_revenue WHERE rank <= 3
""")
q1_sql.show(10)

[Stage 15:===========================================>              (3 + 1) / 4]

+--------------+---------------------+------------------+----+
|customer_state|product_category_name|       total_sales|rank|
+--------------+---------------------+------------------+----+
|            AC|        esporte_lazer|           1677.46|   1|
|            AC|   relogios_presentes|            1389.6|   2|
|            AC|         beleza_saude|           1386.58|   3|
|            AL|         beleza_saude|          12681.26|   1|
|            AL|   relogios_presentes|11605.019999999999|   2|
|            AL| informatica_acess...|7605.6100000000015|   3|
|            AM|         beleza_saude| 2776.030000000001|   1|
|            AM|   ferramentas_jardim|           1980.88|   2|
|            AM| informatica_acess...|1815.7399999999998|   3|
|            AP| informatica_acess...|2050.0299999999997|   1|
+--------------+---------------------+------------------+----+
only showing top 10 rows



## Tính thời gian giao hàng trung bình theo từng tháng và so sánh với thời gian dự kiến để xem tháng nào logistics của Olist bị trễ nhiều nhất

In [9]:
q2_sql = spark.sql("""
    WITH monthly_sales AS (
        SELECT 
            DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') as month,
            SUM(payment_value) as monthly_revenue
        FROM orders o
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
    )
    SELECT 
        month,
        monthly_revenue,
        SUM(monthly_revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
    FROM monthly_sales
    ORDER BY month
""")
q2_sql.show()

26/06/12 14:53:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 14:53:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 14:53:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 14:53:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 14:53:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 14:53:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/12 1

+-------+------------------+--------------------+
|  month|   monthly_revenue|       running_total|
+-------+------------------+--------------------+
|2016-10|          46566.71|            46566.71|
|2016-12|             19.62|            46586.33|
|2017-01|127545.67000000001|            174132.0|
|2017-02|271298.64999999997|  445430.64999999997|
|2017-03|414369.38999999955|   859800.0399999996|
|2017-04|390952.18000000005|  1250752.2199999997|
|2017-05|         567066.73|  1817818.9499999997|
|2017-06| 490225.5999999999|          2308044.55|
|2017-07| 566403.9299999997|  2874448.4799999995|
|2017-08| 646000.6100000003|          3520449.09|
|2017-09| 701169.9900000013|   4221619.080000001|
|2017-10| 751140.2699999999|   4972759.350000001|
|2017-11| 1153528.050000003|   6126287.400000004|
|2017-12|         843199.17|   6969486.570000004|
|2018-01|        1078606.86|   8048093.430000004|
|2018-02| 966510.8799999999|   9014604.310000004|
|2018-03|         1120678.0|1.0135282310000004E7|


## Xác định khoảng cách thời gian (số ngày) giữa 2 lần mua hàng liên tiếp của mỗi khách hàng

In [10]:
q3_sql = spark.sql("""
    WITH customer_purchase_sequences AS (
        SELECT 
            c.customer_unique_id,
            o.order_purchase_timestamp,
            LAG(o.order_purchase_timestamp) OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) as previous_purchase
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'
    )
    SELECT 
        customer_unique_id,
        order_purchase_timestamp,
        previous_purchase,
        DATEDIFF(order_purchase_timestamp, previous_purchase) as days_between_orders
    FROM customer_purchase_sequences
    WHERE previous_purchase IS NOT NULL
""")
q3_sql.show(10)

[Stage 29:===========================================>              (3 + 1) / 4]

+--------------------+------------------------+-------------------+-------------------+
|  customer_unique_id|order_purchase_timestamp|  previous_purchase|days_between_orders|
+--------------------+------------------------+-------------------+-------------------+
|004288347e5e88a27...|     2018-01-14 07:36:54|2017-07-27 14:13:03|                171|
|00a39521eb40f7012...|     2018-06-03 10:12:57|2018-05-23 20:14:21|                 11|
|013ef03e0f3f408dd...|     2018-07-20 04:13:54|2018-06-21 04:46:11|                 29|
|0178b244a5c281fb2...|     2018-07-28 13:13:00|2017-05-10 20:04:09|                444|
|031e19fc630c4121f...|     2018-07-04 01:20:01|2018-07-04 01:20:01|                  0|
|031ea691b99fc101d...|     2017-07-10 22:28:33|2017-05-16 13:42:51|                 55|
|032b3a42598667caf...|     2018-07-24 21:17:34|2018-07-23 15:11:08|                  1|
|0341bbd5c969923a0...|     2018-05-30 13:00:03|2018-05-28 14:18:26|                  2|
|0361e980b28826f4d...|     2018-

## Thống kê tỷ lệ đơn hàng bị giao trễ hạn và số ngày trễ trung bình theo từng tháng

In [11]:
q4_sql = spark.sql("""
    SELECT 
        DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') as purchase_month,
        COUNT(order_id) as total_orders,
        SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) as late_orders,
        ROUND((SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1 ELSE 0 END) / COUNT(order_id)) * 100, 2) as late_rate_percentage,
        ROUND(AVG(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) ELSE NULL END), 1) as avg_days_delayed
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
    GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
    ORDER BY purchase_month
""")
q4_sql.show()

[Stage 32:==============>                                           (1 + 3) / 4]

+--------------+------------+-----------+--------------------+----------------+
|purchase_month|total_orders|late_orders|late_rate_percentage|avg_days_delayed|
+--------------+------------+-----------+--------------------+----------------+
|       2016-09|           1|          1|               100.0|            36.0|
|       2016-10|         265|          3|                1.13|             6.3|
|       2016-12|           1|          0|                 0.0|            NULL|
|       2017-01|         750|         23|                3.07|            19.3|
|       2017-02|        1653|         53|                3.21|            19.2|
|       2017-03|        2546|        142|                5.58|            20.8|
|       2017-04|        2303|        181|                7.86|            10.3|
|       2017-05|        3545|        128|                3.61|            10.4|
|       2017-06|        3135|        121|                3.86|            11.2|
|       2017-07|        3872|        133

## Tìm các khách hàng có tổng chi tiêu cao hơn mức chi tiêu trung bình của chính bang mà họ đang sinh sống

In [13]:
q6_sql = spark.sql("""
    WITH user_spend AS (
        SELECT c.customer_unique_id, c.customer_state, SUM(p.payment_value) as total_spend
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        JOIN payments p ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY c.customer_unique_id, c.customer_state
    ),
    state_avg_spend AS (
        SELECT customer_state, AVG(total_spend) as avg_state_spend
        FROM user_spend
        GROUP BY customer_state
    )
    SELECT u.customer_unique_id, u.customer_state, ROUND(u.total_spend, 2) as customer_spend, ROUND(s.avg_state_spend, 2) as state_average
    FROM user_spend u
    JOIN state_avg_spend s ON u.customer_state = s.customer_state
    WHERE u.total_spend > s.avg_state_spend
    ORDER BY u.customer_state, u.total_spend DESC
""")
q6_sql.show(10)

+--------------------+--------------+--------------+-------------+
|  customer_unique_id|customer_state|customer_spend|state_average|
+--------------------+--------------+--------------+-------------+
|62a459e5629b03dd7...|            AC|        1251.7|       257.71|
|086d6b5b5ba195a91...|            AC|        995.18|       257.71|
|3947ca729a860c522...|            AC|        905.93|       257.71|
|3e5c928acf49c4b95...|            AC|        861.26|       257.71|
|28989ef45087c96e5...|            AC|        723.15|       257.71|
|12e92c0f870fc6941...|            AC|        646.44|       257.71|
|22c739518f5240ffd...|            AC|         618.6|       257.71|
|0845d810b482f4d46...|            AC|        595.49|       257.71|
|c6bc2bf4b75f3f9da...|            AC|        591.88|       257.71|
|2ec67750cd5b98553...|            AC|        548.93|       257.71|
+--------------------+--------------+--------------+-------------+
only showing top 10 rows



## Đánh giá mối tương quan giữa số kỳ trả góp (Installments) và điểm đánh giá (Review Score)

In [14]:
q8_sql = spark.sql("""
    SELECT 
        p.payment_installments,
        COUNT(DISTINCT o.order_id) as total_orders,
        ROUND(AVG(r.review_score), 2) as avg_review_score
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    JOIN reviews r ON o.order_id = r.order_id
    WHERE p.payment_type = 'credit_card'
    GROUP BY p.payment_installments
    HAVING COUNT(DISTINCT o.order_id) > 100
    ORDER BY p.payment_installments
""")
q8_sql.show()

+--------------------+------------+----------------+
|payment_installments|total_orders|avg_review_score|
+--------------------+------------+----------------+
|                   1|       25238|            4.15|
|                   2|       12293|            4.11|
|                   3|       10355|            4.06|
|                   4|        7042|            4.05|
|                   5|        5188|            4.06|
|                   6|        3881|            4.08|
|                   7|        1607|            4.04|
|                   8|        4222|            4.02|
|                   9|         634|            4.08|
|                  10|        5261|            3.96|
|                  12|         131|            3.95|
+--------------------+------------+----------------+



In [15]:
q8_sql.explain(True)

== Parsed Logical Plan ==
'Sort ['p.payment_installments ASC NULLS FIRST], true
+- 'UnresolvedHaving ('COUNT(distinct 'o.order_id) > 100)
   +- 'Aggregate ['p.payment_installments], ['p.payment_installments, 'COUNT(distinct 'o.order_id) AS total_orders#528, 'ROUND('AVG('r.review_score), 2) AS avg_review_score#529]
      +- 'Filter ('p.payment_type = credit_card)
         +- 'Join Inner, ('o.order_id = 'r.order_id)
            :- 'Join Inner, ('o.order_id = 'p.order_id)
            :  :- 'SubqueryAlias o
            :  :  +- 'UnresolvedRelation [orders], [], false
            :  +- 'SubqueryAlias p
            :     +- 'UnresolvedRelation [payments], [], false
            +- 'SubqueryAlias r
               +- 'UnresolvedRelation [reviews], [], false

== Analyzed Logical Plan ==
payment_installments: int, total_orders: bigint, avg_review_score: double
Sort [payment_installments#84 ASC NULLS FIRST], true
+- Filter (total_orders#528L > cast(100 as bigint))
   +- Aggregate [payment_installm